In [ ]:
# Cell 1
import pandas as pd
import numpy as np

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedShuffleSplit, ParameterGrid
from sklearn.metrics import (
    precision_score,
    recall_score,
    roc_auc_score,
    accuracy_score,
    f1_score
)
from imblearn.over_sampling import RandomOverSampler

In [ ]:
# Cell 2
# Validation settings
n_splits = 5
seed = 10

# Stratified CV to preserve class distribution
kf = StratifiedShuffleSplit(
    n_splits=n_splits,
    test_size=1 / n_splits,
    random_state=seed
)

In [ ]:
# Cell 3
# Data preparation
def prepare_fold_data(X_train, X_val):
    X_train = X_train.copy()
    X_val = X_val.copy()

    # Convert boolean features to numeric
    bool_cols_train = X_train.select_dtypes(include=["bool"]).columns
    bool_cols_val = X_val.select_dtypes(include=["bool"]).columns

    if len(bool_cols_train) > 0:
        X_train[bool_cols_train] = X_train[bool_cols_train].astype(int)
    if len(bool_cols_val) > 0:
        X_val[bool_cols_val] = X_val[bool_cols_val].astype(int)

    # Dummy encoding + align feature space
    X_train = pd.get_dummies(X_train, drop_first=False)
    X_val = pd.get_dummies(X_val, drop_first=False)

    X_train, X_val = X_train.align(X_val, join="left", axis=1, fill_value=0)

    return X_train, X_val

In [ ]:
# Cell 4
# Evaluate one set of tree parameters
def evaluate_tree_params(prefix, params, upsample=True):
    X = pd.read_csv(f"data/{prefix}_X_train.csv", keep_default_na=False)
    y = pd.read_csv(f"data/{prefix}_y_train.csv", keep_default_na=False).values.ravel()

    cv_scores = {
        "fold": [],
        "test_precision": [],
        "test_recall": [],
        "test_roc_auc": [],
        "test_accuracy": [],
        "test_f1": []
    }

    splits = list(kf.split(X, y))

    for fold_idx, (train_idx, val_idx) in enumerate(splits):
        X_train = X.iloc[train_idx].copy()
        y_train = y[train_idx].copy()

        X_val = X.iloc[val_idx].copy()
        y_val = y[val_idx].copy()

        # Oversampling applied only on training fold
        if upsample:
            upsampler = RandomOverSampler(random_state=seed)
            X_train, y_train = upsampler.fit_resample(X_train, y_train)

            if not isinstance(X_train, pd.DataFrame):
                X_train = pd.DataFrame(X_train, columns=X.columns)

        X_train_prepared, X_val_prepared = prepare_fold_data(X_train, X_val)

        clf = DecisionTreeClassifier(
            random_state=seed,
            **params
        )
        clf.fit(X_train_prepared, y_train)

        y_pred = clf.predict(X_val_prepared)
        y_prob = clf.predict_proba(X_val_prepared)[:, 1]

        cv_scores["fold"].append(fold_idx + 1)
        cv_scores["test_precision"].append(precision_score(y_val, y_pred))
        cv_scores["test_recall"].append(recall_score(y_val, y_pred))
        cv_scores["test_roc_auc"].append(roc_auc_score(y_val, y_prob))
        cv_scores["test_accuracy"].append(accuracy_score(y_val, y_pred))
        cv_scores["test_f1"].append(f1_score(y_val, y_pred))

    # Convert to arrays for summary statistics
    for key in ["test_precision", "test_recall", "test_roc_auc", "test_accuracy", "test_f1"]:
        cv_scores[key] = np.array(cv_scores[key])

    # Return mean performance for this parameter setting
    summary = {
        "params": params,
        "precision_mean": cv_scores["test_precision"].mean(),
        "recall_mean": cv_scores["test_recall"].mean(),
        "roc_auc_mean": cv_scores["test_roc_auc"].mean(),
        "accuracy_mean": cv_scores["test_accuracy"].mean(),
        "f1_mean": cv_scores["test_f1"].mean()
    }

    return cv_scores, summary

In [ ]:
# Cell 5
# Hyperparameter grid
tree_param_grid = {
    "max_depth": [3, 5, 7, 10, 15],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10],
    "min_weight_fraction_leaf": [0.0, 0.01, 0.02],
    "criterion": ["gini", "entropy"],
    "splitter": ["best"],
    "min_impurity_decrease": [0.0, 0.0001, 0.001, 0.01],
    "class_weight": [None]
}

In [ ]:
# Cell 6
# Grid search over parameters
def tune_classification_tree(prefix, upsample=True):
    all_results = []

    # Evaluate all parameter combinations
    for params in ParameterGrid(tree_param_grid):
        _, summary = evaluate_tree_params(prefix, params, upsample=upsample)
        all_results.append(summary)

    results_df = pd.DataFrame(all_results)
    # Sort by performance
    results_df = results_df.sort_values(
        by=["roc_auc_mean", "f1_mean", "accuracy_mean"],
        ascending=False
    ).reset_index(drop=True)

    best_row = results_df.iloc[0]

    best_params = best_row["params"]

    print(f"Best params for {prefix}:")
    print(best_params)
    print("Best mean ROC AUC:", round(best_row["roc_auc_mean"], 3))
    print("Best mean precision:", round(best_row["precision_mean"], 3))
    print("Best mean recall:", round(best_row["recall_mean"], 3))
    print("Best mean accuracy:", round(best_row["accuracy_mean"], 3))
    print("Best mean F1:", round(best_row["f1_mean"], 3))

    return results_df, best_params

In [ ]:
# Cell 7
# Evaluate best model with CV + SE
def cross_validate_best_tree(prefix, best_params, upsample=True):
    cv_scores, _ = evaluate_tree_params(prefix, best_params, upsample=upsample)

    summary_df = pd.DataFrame({
        "fold": cv_scores["fold"],
        "precision": cv_scores["test_precision"],
        "recall": cv_scores["test_recall"],
        "roc_auc": cv_scores["test_roc_auc"],
        "accuracy": cv_scores["test_accuracy"],
        "f1": cv_scores["test_f1"]
    })

    # Mean performance
    mean_row = pd.DataFrame([{
        "fold": "mean",
        "precision": summary_df["precision"].mean(),
        "recall": summary_df["recall"].mean(),
        "roc_auc": summary_df["roc_auc"].mean(),
        "accuracy": summary_df["accuracy"].mean(),
        "f1": summary_df["f1"].mean()
    }])

    # Cross-validated SE
    n = n_splits

    se_row = pd.DataFrame([{
        "fold": "se",
        "precision": summary_df["precision"].std(ddof=1) / np.sqrt(n),
        "recall": summary_df["recall"].std(ddof=1) / np.sqrt(n),
        "roc_auc": summary_df["roc_auc"].std(ddof=1) / np.sqrt(n),
        "accuracy": summary_df["accuracy"].std(ddof=1) / np.sqrt(n),
        "f1": summary_df["f1"].std(ddof=1) / np.sqrt(n)
    }])

    summary_df = pd.concat([summary_df, mean_row, se_row], ignore_index=True)

    print(f"Results for {prefix} (upsample={upsample})")

    for metric in ["precision", "recall", "roc_auc", "accuracy", "f1"]:
        mean_value = summary_df.loc[summary_df["fold"] == "mean", metric].values[0]
        se_value = summary_df.loc[summary_df["fold"] == "se", metric].values[0]
        print(f"Mean {metric}: {mean_value:.3f} ± {se_value:.3f}")

    return cv_scores, summary_df

In [ ]:
# Cell 8
tree_results_mask_before, best_tree_params_mask_before = tune_classification_tree("mask_before", upsample=True)
tree_results_mask_before.head()

Best params for mask_before:
{'class_weight': None, 'criterion': 'gini', 'max_depth': 15, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.01, 'splitter': 'best'}
Best mean ROC AUC: 0.814
Best mean precision: 0.49
Best mean recall: 0.727
Best mean accuracy: 0.733
Best mean F1: 0.585


,params,precision_mean,recall_mean,roc_auc_mean,accuracy_mean,f1_mean
0,"{'class_weight': None, 'criterion': 'gini', 'm...",0.489509,0.726868,0.814426,0.732564,0.584975
1,"{'class_weight': None, 'criterion': 'gini', 'm...",0.489509,0.726868,0.814426,0.732564,0.584975
2,"{'class_weight': None, 'criterion': 'gini', 'm...",0.489509,0.726868,0.814426,0.732564,0.584975
3,"{'class_weight': None, 'criterion': 'gini', 'm...",0.489509,0.726868,0.814426,0.732564,0.584975
4,"{'class_weight': None, 'criterion': 'gini', 'm...",0.489509,0.726868,0.814426,0.732564,0.584975


In [ ]:
# Cell 9
cv_scores_tree_mask_before, summary_tree_mask_before = cross_validate_best_tree(
    "mask_before",
    best_tree_params_mask_before,
    upsample=True
)

summary_tree_mask_before

Results for mask_before (upsample=True)
Mean precision: 0.490 ± 0.004
Mean recall: 0.727 ± 0.005
Mean roc_auc: 0.814 ± 0.003
Mean accuracy: 0.733 ± 0.003
Mean f1: 0.585 ± 0.003


,fold,precision,recall,roc_auc,accuracy,f1
0,1,0.479452,0.723370,0.816114,0.724650,0.576679
1,2,0.483158,0.729730,0.809224,0.727535,0.581381
2,3,0.497338,0.742448,0.824742,0.738664,0.595663
3,4,0.497778,0.712242,0.807950,0.739077,0.586004
4,5,0.489818,0.726550,0.814099,0.732894,0.585147
5,mean,0.489509,0.726868,0.814426,0.732564,0.584975
6,se,0.003683,0.004885,0.002986,0.002895,0.003139


In [ ]:
# Cell 10
tree_results_mask_after, best_tree_params_mask_after = tune_classification_tree("mask_after", upsample=True)
cv_scores_tree_mask_after, summary_tree_mask_after = cross_validate_best_tree(
    "mask_after",
    best_tree_params_mask_after,
    upsample=True
)

tree_results_general_before, best_tree_params_general_before = tune_classification_tree("general_before", upsample=True)
cv_scores_tree_general_before, summary_tree_general_before = cross_validate_best_tree(
    "general_before",
    best_tree_params_general_before,
    upsample=True
)

tree_results_general_after, best_tree_params_general_after = tune_classification_tree("general_after", upsample=True)
cv_scores_tree_general_after, summary_tree_general_after = cross_validate_best_tree(
    "general_after",
    best_tree_params_general_after,
    upsample=True
)

Best params for mask_after:
{'class_weight': None, 'criterion': 'entropy', 'max_depth': 15, 'min_impurity_decrease': 0.001, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'splitter': 'best'}
Best mean ROC AUC: 0.864
Best mean precision: 0.889
Best mean recall: 0.854
Best mean accuracy: 0.821
Best mean F1: 0.871
Results for mask_after (upsample=True)
Mean precision: 0.889 ± 0.002
Mean recall: 0.854 ± 0.005
Mean roc_auc: 0.864 ± 0.003
Mean accuracy: 0.821 ± 0.004
Mean f1: 0.871 ± 0.003
Best params for general_before:
{'class_weight': None, 'criterion': 'entropy', 'max_depth': 15, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.01, 'splitter': 'best'}
Best mean ROC AUC: 0.741
Best mean precision: 0.703
Best mean recall: 0.681
Best mean accuracy: 0.678
Best mean F1: 0.691
Results for general_before (upsample=True)
Mean precision: 0.703 ± 0.007
Mean recall: 0.681 ± 0.011
Mean roc_auc: 0.741 ± 0.006
M

In [ ]:
# Cell 11
# Build comparison table
def get_mean_se(summary_df, metric):
    mean_value = summary_df.loc[summary_df["fold"] == "mean", metric].values[0]
    se_value = summary_df.loc[summary_df["fold"] == "se", metric].values[0]
    return mean_value, se_value


summaries = {
    "mask_before": summary_tree_mask_before,
    "mask_after": summary_tree_mask_after,
    "general_before": summary_tree_general_before,
    "general_after": summary_tree_general_after
}

rows = []

for model_name, summary in summaries.items():
    row = {"model": model_name}
    
    for metric in ["precision", "recall", "roc_auc", "accuracy", "f1"]:
        mean_value, se_value = get_mean_se(summary, metric)
        row[metric] = mean_value
        row[f"{metric}_se"] = se_value
        row[f"{metric}_mean_se"] = f"{mean_value:.3f} ± {se_value:.3f}"
    
    rows.append(row)

comparison_df = pd.DataFrame(rows)

comparison_df

,model,precision,precision_se,precision_mean_se,recall,recall_se,recall_mean_se,roc_auc,roc_auc_se,roc_auc_mean_se,accuracy,accuracy_se,accuracy_mean_se,f1,f1_se,f1_mean_se
0,mask_before,0.489509,0.003683,0.490 ± 0.004,0.726868,0.004885,0.727 ± 0.005,0.814426,0.002986,0.814 ± 0.003,0.732564,0.002895,0.733 ± 0.003,0.584975,0.003139,0.585 ± 0.003
1,mask_after,0.889064,0.001748,0.889 ± 0.002,0.853903,0.005325,0.854 ± 0.005,0.864188,0.003258,0.864 ± 0.003,0.821425,0.003992,0.821 ± 0.004,0.871100,0.003190,0.871 ± 0.003
2,general_before,0.702812,0.007387,0.703 ± 0.007,0.681182,0.010611,0.681 ± 0.011,0.741346,0.005874,0.741 ± 0.006,0.677906,0.004241,0.678 ± 0.004,0.691458,0.004515,0.691 ± 0.005
3,general_after,0.877821,0.003143,0.878 ± 0.003,0.704882,0.011798,0.705 ± 0.012,0.797565,0.002972,0.798 ± 0.003,0.714943,0.006398,0.715 ± 0.006,0.781623,0.006646,0.782 ± 0.007


In [ ]:
# Cell 12
pd.DataFrame(comparison_df).to_csv("results/tree_mask_before.csv", index=False)